# Mini-Project | Natural Language Processing | Detecting fake/real news 

## 1. - Import libraries and read files

In [1]:
# pip uninstall transformers -y

In [2]:
# pip install transformers

In [3]:
import pandas as pd
import numpy as np
import string
import re
import html
from bs4 import BeautifulSoup

from nltk import word_tokenize, pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import LinearSVC

import unicodedata

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

from datasets import Dataset


In [4]:
df = pd.read_csv(f"dataset/training_data.csv", delimiter='\t', header=None, names=["label","text"])

In [5]:
rows = df.shape[0]          # checking for the number of rows in the dataset
columns = df.shape[1]       # checking for the number of columns in the dataset

print(f"We have \033[1m{rows}\033[0m rows in the 'training_data.csv'")
print(f"We have \033[1m{columns}\033[0m columns in the 'training_data.csv'")

We have 34152 rows in the 'training_data.csv'
We have 2 columns in the 'training_data.csv'


In [6]:
print(f"We have \033[1m{df["label"][df["label"] == 1].count()}\033[0m real news in the 'training_data.csv'")
display(df[df["label"] == 1].head())      # show the first headlines that are real news


We have 16580 real news in the 'training_data.csv'


,label,text
17572,1,as u.s. budget fight looms\trepublicans flip t...
17573,1,u.s. military to accept transgender recruits o...
17574,1,senior u.s. republican senator: 'let mr. muell...
17575,1,fbi russia probe helped by australian diplomat...
17576,1,trump wants postal service to charge 'much mor...


In [7]:
print(f"We have \033[1m{df["label"][df["label"] == 0].count()}\033[0m fake news in the 'training_data.csv'")
display(df[df["label"] == 0].head())     # show the first headlines that are fake news

We have 17572 fake news in the 'training_data.csv'


,label,text
0,0,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...


## 2. - Data preprocessing

### 2.1 - Define different functions

In [8]:
# Remove stop words
def remove_words(text):
    stop_words = set(stopwords.words('english'))
    pattern = r"\b(" + "|".join(map(re.escape, stop_words)) + r")\b"
    return re.sub(pattern, "", text, flags=re.IGNORECASE)

#############################################################
# convert Unicode to ASCII-similiar
def normalize_unicode(s: str) -> str:
    s = str(s)
    # NFKC/NFKD glätten viele vollbreite/typografische Varianten
    return unicodedata.normalize("NFKC", s)

# Remove all the special characters
def remove_punctuation_unicode(s: str) -> str:
    s = str(s)
    return "".join(ch for ch in s
                   if not unicodedata.category(ch).startswith("P"))

#############################################################
# Remove numbers
def remove_numbers(text):
    return re.sub(r"\d+", "", text)                 # one or more digits

#############################################################
# Remove all single characters
def remove_single_chars(text):
    return re.sub(r"(?<=\s).(?=\s)", "", text)      # exactly one single character, if there is a space before and after this character

#############################################################
# Remove single characters from the start
def remove_first_char(text):
    return re.sub(r"^(?=\S\s)\S", "", text)         # remove first char if it is followed by a space

#############################################################
# Substitute multiple spaces with single space

def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()        # replace multiple spaces by only a single space. .striß -> Delete spaces at the beginning or the end

#############################################################
# Remove prefixed 'b'
def remove_b_before_capital(text):
    return re.sub(r"b(?=[A-Z])", "", text)          # delete "b" if it is followed by a capital letter

#############################################################
# Convert to Lowercase
def to_lowercase(text):
    return text.lower()

#############################################################
# get POS TAG
def to_wordnet_pos(treebank_tag: str):
    if not treebank_tag:
        return wordnet.NOUN
    t = treebank_tag[0]
    return {
        'J': wordnet.ADJ,   # JJ, JJR, JJS
        'V': wordnet.VERB,  # VB, VBD, VBG, VBN, VBP, VBZ
        'N': wordnet.NOUN,  # NN, NNS, NNP, NNPS
        'R': wordnet.ADV    # RB, RBR, RBS
    }.get(t, wordnet.NOUN)

#############################################################
# Lemmatize with POS TAG
def lemmatize_with_pos(text: str) -> str:
    lemma = WordNetLemmatizer()
    tokens = word_tokenize(text)
    tagged  = pos_tag(tokens)
    lemmas = [lemma.lemmatize(tok, pos=to_wordnet_pos(tag)) for tok, tag in tagged]
    return re.sub(r"\s+", " ", " ".join(lemmas)).strip()

#############################################################
def clean_data(data):
    data = str(data)
    data = normalize_unicode(data)
    data = remove_punctuation_unicode(data) 
    data = remove_numbers(data) 
    data = remove_single_chars(data) 
    data = remove_first_char(data) 
    data = normalize_spaces(data) 
    data = remove_b_before_capital(data) 
    data = to_lowercase(data) 
    data = remove_words(data) 
    data = normalize_spaces(data) 
    data = lemmatize_with_pos(data) 
    return data

In [9]:
stop_words = set(stopwords.words('english'))
print(stop_words)

{'any', 'which', "it'd", 'while', 'above', "should've", 'off', "haven't", 'to', 'or', "that'll", 'both', "they're", 'up', 'on', 'i', "hadn't", "i've", 'me', 'whom', 'why', 'by', 'because', "isn't", "won't", 'these', 'wasn', 'having', 'ain', 'shan', 'm', 'out', 'had', 'doesn', 'only', 'of', 'nor', "aren't", 'it', 'doing', 'its', 'then', 'during', 'ours', 'her', "i'm", 'weren', 'their', "they'll", 'him', 'yourselves', 'does', 'from', 'before', 're', 'few', "we'll", "they've", 'have', 'with', 'itself', 'so', "don't", 'than', 'and', "i'll", 'were', 'other', 'once', 't', 'aren', 'haven', 'as', 'we', 'about', 'your', 'yours', 'shouldn', 'each', 'couldn', 'an', 'are', 'how', 'she', 'herself', 'same', 'himself', "she's", "wasn't", 'am', 'the', "couldn't", 'below', "you'd", "we've", 'against', 'has', 'most', "he'll", 'you', 'what', 'hadn', 'when', 'being', 'such', "it'll", "you'll", 'very', 'was', "it's", 'more', 'y', "weren't", 'where', 'd', 'for', "she'd", 'yourself', 's', 'further', 'theirs'

### 2.2 - split data (train/test)

In [10]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ensures balanced labels
)

### 2.3 - cleaning the data

In [11]:
X_train_clean = X_train.apply(clean_data)
X_test_clean = X_test.apply(clean_data)

In [12]:
print(X_train_clean.head(),"\n")
print(X_test_clean.head())

6851     republican punish georgia governor refuse lega...
17313    father soldier slain niger defend president tr...
22435    south dakota governor veto loosen conceal carr...
29488    turkey erdogan say take jerusalem resolution u...
6625     bill maher insult trump suppose masculinity gl...
Name: text, dtype: object 

10145    msnbc propagandist word trump modern day swast...
26343        clinton say trump divisive candidate lifetime
22173     ivanka trump becomes unpaid white house employee
365      trump supporter mother rally massively outnumb...
13323    break fresno police release graphic video fata...
Name: text, dtype: object


In [13]:
df["text"][10145]       # comparing some headlines with the one before cleaning

'msnbc propagandist: the word ‚trump‚ is ‚modern day swastika‚‚blames trump for mosque arsons [video]'

## 3. - Creating text vectors

### 3.1 - Bag of Words using CountVectotizer

In [14]:
real = X_train_clean[y_train == 1]

vectorizer = CountVectorizer(ngram_range=(1,1), min_df=1)    
X = vectorizer.fit_transform(real)                          # sparse CSR

freqs = np.asarray(X.sum(axis=0)).ravel()                   # fast sum function
vocab = vectorizer.get_feature_names_out()

dict_of_words = dict(zip(vocab, freqs.astype(int)))
print(f"number of words in dictionary: {len(dict_of_words)}\n")

top10_real = sorted(dict_of_words.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("top 10 used words in real news:")
display(pd.DataFrame(top10_real,columns=["word","count"]))

#####################################################################################
fake = X_train_clean[y_train == 0]

vectorizer = CountVectorizer(ngram_range=(1,1), min_df=1)   
X = vectorizer.fit_transform(fake)                          # sparse CSR

freqs = np.asarray(X.sum(axis=0)).ravel()                   # fast sum function
vocab = vectorizer.get_feature_names_out()

dict_of_words = dict(zip(vocab, freqs.astype(int)))

top10_fake = sorted(dict_of_words.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("top 10 used words in fake news:")
display(pd.DataFrame(top10_fake,columns=["word","count"]))

number of words in dictionary: 9171

top 10 used words in real news:


,word,count
0,trump,4075
1,say,1969
2,house,1151
3,republican,691
4,white,655
5,russia,591
6,senate,580
7,new,538
8,bill,527
9,clinton,500


top 10 used words in fake news:


,word,count
0,trump,5556
1,video,4384
2,hillary,1092
3,obama,1045
4,get,795
5,clinton,660
6,break,647
7,president,610
8,say,590
9,tweet,581


### 3.2 - TF-IDF (Term Frequency - Inverse Document Frequency)

In [15]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(X_train_clean)
feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()

# checking the word with the highest TF-IDF value in a specified headline
news_number = tfidf_array[800]
index = np.argmax(news_number)
value = news_number[index]
display(f"word: {feature_names[index]} -> value: {value}")

'word: leading -> value: 0.5327027332809747'

## 4. - Testing some classifiers

### 4.1 - Logistic Regression -> !!! working better with uncleaned data !!!

In [37]:
tfidf_lr = Pipeline([
    ("vec", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=1,
        max_df=0.98,
        sublinear_tf=True        
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
y_proba = tfidf_lr.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9424681598594642
              precision    recall  f1-score   support

           0      0.949     0.939     0.944      3515
           1      0.936     0.946     0.941      3316

    accuracy                          0.942      6831
   macro avg      0.942     0.943     0.942      6831
weighted avg      0.943     0.942     0.942      6831

Confusion Matrix:
 [[3300  215]
 [ 178 3138]]


#### 4.1.1 - with CountVectorizer -> !!! working better with uncleaned data !!! -> +1% accuracy

In [53]:
tfidf_lr = Pipeline([
    ("vec", CountVectorizer(
        ngram_range=(1,2),
        min_df=1        
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9483238178890353
              precision    recall  f1-score   support

           0      0.962     0.937     0.949      3515
           1      0.935     0.960     0.947      3316

    accuracy                          0.948      6831
   macro avg      0.948     0.949     0.948      6831
weighted avg      0.949     0.948     0.948      6831

Confusion Matrix:
 [[3293  222]
 [ 131 3185]]


### 4.2 - RandomForestClassifier -> !!! working better with uncleaned data !!! -> +1% accuracy

In [44]:
tfidf_lr = Pipeline([
    ("vec", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True        
    )),
    ("rfc", RandomForestClassifier(
        n_estimators=1000,
        max_depth=None,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9369052847313717
              precision    recall  f1-score   support

           0      0.960     0.915     0.937      3515
           1      0.914     0.960     0.937      3316

    accuracy                          0.937      6831
   macro avg      0.937     0.938     0.937      6831
weighted avg      0.938     0.937     0.937      6831

Confusion Matrix:
 [[3217  298]
 [ 133 3183]]


#### 4.2.1 - with CountVectorizer

In [54]:
tfidf_lr = Pipeline([
    ("vec", CountVectorizer(
        ngram_range=(1,2),
        min_df=1        
    )),
    ("rfc", RandomForestClassifier(
        n_estimators=1000,
        max_depth=None,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    

Accuracy: 0.9391011564924608
              precision    recall  f1-score   support

           0      0.937     0.946     0.941      3515
           1      0.942     0.932     0.937      3316

    accuracy                          0.939      6831
   macro avg      0.939     0.939     0.939      6831
weighted avg      0.939     0.939     0.939      6831

Confusion Matrix:
 [[3324  191]
 [ 225 3091]]


#### 4.2.2 - Optimization -> !!! working better with uncleaned data !!! -> +2% accuracy

In [146]:
pipeline = Pipeline([
    ("vec", TfidfVectorizer(sublinear_tf=True, lowercase=True)),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid = {
    "vec__ngram_range": [(1,1), (1,2)],
    "vec__min_df": [2, 3, 5],
    "vec__max_df": [0.9, 0.95],
    "vec__max_features": [100000, 200000],
    
    "clf__n_estimators": [300, 500, 1000],
    "clf__max_depth": [None, 50],
    "clf__max_features": ["sqrt", "log2"],
    "clf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best F1:", grid.best_score_)

Fitting 3 folds for each of 576 candidates, totalling 1728 fits
Best parameters: {'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_split': 2, 'clf__n_estimators': 500, 'vec__max_df': 0.9, 'vec__max_features': 100000, 'vec__min_df': 2, 'vec__ngram_range': (1, 1)}
Best F1: 0.9383981533549663


In [ ]:
tfidf_lr = Pipeline([
    ("vec", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9,
        max_features=100000,
        sublinear_tf=True        
    )),
    ("rfc", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        max_features='log2',
        min_samples_split=2,
        # class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9500805152979066
              precision    recall  f1-score   support

           0      0.973     0.929     0.950      3515
           1      0.928     0.973     0.950      3316

    accuracy                          0.950      6831
   macro avg      0.950     0.951     0.950      6831
weighted avg      0.951     0.950     0.950      6831

Confusion Matrix:
 [[3265  250]
 [  91 3225]]


##### 4.2.2.1 - with CountVectorizer

In [58]:
tfidf_lr = Pipeline([
    ("vec", CountVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9,
        max_features=100000      
    )),
    ("rfc", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        max_features='log2',
        min_samples_split=2,
        # class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))  

Accuracy: 0.9506660811008637
              precision    recall  f1-score   support

           0      0.957     0.947     0.952      3515
           1      0.944     0.955     0.949      3316

    accuracy                          0.951      6831
   macro avg      0.951     0.951     0.951      6831
weighted avg      0.951     0.951     0.951      6831

Confusion Matrix:
 [[3328  187]
 [ 150 3166]]


### 4.3 - SVM (Support Vector Machine) -> !!! working better with uncleaned data !!! -> +1%

In [43]:
tfidf_lr = Pipeline([
    ("vec", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True        
    )),
    ("clf", LinearSVC(
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9538866930171278
              precision    recall  f1-score   support

           0      0.957     0.953     0.955      3515
           1      0.950     0.955     0.953      3316

    accuracy                          0.954      6831
   macro avg      0.954     0.954     0.954      6831
weighted avg      0.954     0.954     0.954      6831

Confusion Matrix:
 [[3349  166]
 [ 149 3167]]


#### 4.3.1 - with CountVectorizer

In [ ]:
tfidf_lr = Pipeline([
    ("vec", CountVectorizer(
        ngram_range=(1,2),
        min_df=1,
        max_df=0.9       
    )),
    ("clf", LinearSVC(
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred = tfidf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


## 5. Transformer...

In [16]:
# load pretrained model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [18]:
X_train

6851     republicans punish georgia governor for refusi...
17313    father of soldier slain in niger defends presi...
22435    south dakota's governor vetoes loosening of co...
29488    turkey's erdogan says will take jerusalem reso...
6625     bill maher insults trump‚s supposed masculinit...
                               ...                        
5267     trump sends hillary a pathetic threat just one...
13642    breaking: fbi investigating clinton crony virg...
19391    new russian envoy describes 'warm' meeting wit...
883                                            like father
3939     the hilariously amazing video we‚ve been waiti...
Name: text, Length: 27321, dtype: object

In [19]:
# tokenize the train and test data
X_train_token = tokenizer(
    list(X_train.astype(str)),
    truncation=True,
    padding=True,
    max_length=256
)

X_test_token = tokenizer(
    list(X_test.astype(str)),
    truncation=True,
    padding=True,
    max_length=256
)

# add labels to the tokonized data
train_dataset = {
    "input_ids": X_train_token["input_ids"],
    "attention_mask": X_train_token["attention_mask"],
    "labels": np.array(y_train)
}

eval_dataset = {
    "input_ids": X_test_token["input_ids"],
    "attention_mask": X_test_token["attention_mask"],
    "labels": np.array(y_test)
}

# convert to HuggingFace-dataset
train_ds = Dataset.from_dict(train_dataset)
eval_ds = Dataset.from_dict(eval_dataset)

In [27]:
training_args = TrainingArguments(
    output_dir="results2",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # oder z.B. "accuracy"
    greater_is_better=False,            # für


    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()

c:\Users\Mayo-Laptop\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
500,0.044880,0.081338
1000,0.019754,0.102933
1500,0.041116,0.083850
2000,0.000204,0.101178
2500,0.002866,0.124086
3000,0.015318,0.087733
3500,0.000034,0.109252
4000,0.007001,0.111753
4500,0.000015,0.110051
5000,0.000014,0.105730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Mayo-Laptop\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

c:\Users\Mayo-Laptop\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5124, training_loss=0.012279436293784194, metrics={'train_runtime': 8752.9953, 'train_samples_per_second': 9.364, 'train_steps_per_second': 0.585, 'total_flos': 1335972265528572.0, 'train_loss': 0.012279436293784194, 'epoch': 3.0})

In [31]:
best = "results2/checkpoint-500"
model = AutoModelForSequenceClassification.from_pretrained(best)
tokenizer = AutoTokenizer.from_pretrained(best)
trainer = Trainer(model=model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [32]:
pred = trainer.predict(eval_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print("Validation Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


c:\Users\Mayo-Laptop\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Validation Accuracy: 0.9821402430098082
              precision    recall  f1-score   support

           0      0.982     0.983     0.983      3515
           1      0.982     0.981     0.982      3316

    accuracy                          0.982      6831
   macro avg      0.982     0.982     0.982      6831
weighted avg      0.982     0.982     0.982      6831

Confusion Matrix:
 [[3457   58]
 [  64 3252]]


## load best model and predict classes for testing_data.csv

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import torch

model_path = "best_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [9]:
df = pd.read_csv("dataset/testing_data.csv", delimiter='\t', header=None, names=["label","text"])
df.head()

,label,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes
3,2,mi school sends welcome back packet warning ki...
4,2,u.n. seeks 'massive' aid boost amid rohingya '...


In [10]:

if "text" not in df.columns:
    raise ValueError("column 'text' is missing in the csv-file. please check column names.")
df["text"] = df["text"].fillna("").astype(str)

@torch.no_grad()
def predict(text: str) -> int:
    # tokenisation
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,         
        max_length=256        
    )
    # **DistilBERT-Fix**: delete token_type_ids, if available
    inputs.pop("token_type_ids", None)

    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return pred

# make predictions
df["prediction"] = df["text"].apply(predict)

# -> Variante B: overwrite only old class "2"
if "label" in df.columns:
    df.loc[df["label"] == 2, "label"] = df["prediction"]


In [ ]:
# save csv
df.to_csv("dataset/testing_data_with_predictions.csv", index=False)
df.head()

,label,text,prediction
0,2,copycat muslim terrorist arrested with assault...,0
1,2,wow! chicago protester caught on camera admits...,0
2,2,germany's fdp look to fill schaeuble's big shoes,1
3,2,mi school sends welcome back packet warning ki...,0
4,2,u.n. seeks 'massive' aid boost amid rohingya '...,1
